# LeetCode 300 — Longest Increasing Subsequence bằng Python

Link đề: https://leetcode.com/problems/longest-increasing-subsequence/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Binary Search

Skeleton LeetCode:

```python
class Solution:
    def lengthOfLIS(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ file đề có sẵn (bài này không phải Premium). Mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với brute force độc lập (thử mọi tập con bằng mặt nạ bit).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **độ dài** của **dãy con tăng ngặt dài nhất** (Longest Increasing Subsequence — LIS) của mảng; **dãy con (subsequence)** là dãy lấy từ mảng bằng cách **xóa một số phần tử** (có thể không xóa) mà **giữ nguyên thứ tự** — khác **mảng con (subarray)** liên tiếp.
2. Đi từ **vét cạn mọi dãy con** `O(2^n)` → **đệ quy có nhớ** `O(n²)` → **quy hoạch động `dp[i]` = LIS kết thúc tại `i`** `O(n²)` → **"patience sorting": mảng `tails` + `bisect_left`** `O(n log n)` → **cây Fenwick (BIT) trên giá trị đã nén** `O(n log n)`.
3. Hiểu **ý nghĩa của `tails[k]`** ("đuôi nhỏ nhất của một dãy con tăng độ dài `k+1`") và vì sao `tails` luôn **tăng ngặt** nên chặt nhị phân được.
4. Phân biệt `bisect_left` (cho dãy **tăng ngặt**) và `bisect_right` (cho dãy **không giảm**), và nhớ rằng `tails` **không phải** là một LIS thật.
5. Tránh các lỗi kinh điển: nhầm dãy con với mảng con, dùng `bisect_right`, khởi tạo `dp` bằng `0`, quên lấy `max(dp)`, và coi `tails` là đáp án dãy.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`, trả về **độ dài của dãy con tăng ngặt dài nhất**.

**Ví dụ 1**

- Input: `nums = [10,9,2,5,3,7,101,18]`
- Output: `4`
- Giải thích: dãy con tăng dài nhất là `[2,3,7,101]`, độ dài `4`.

**Ví dụ 2**

- Input: `nums = [0,1,0,3,2,3]`
- Output: `4`

**Ví dụ 3**

- Input: `nums = [7,7,7,7,7,7,7]`
- Output: `1`

**Ràng buộc:** `1 <= nums.length <= 2500`; `-10^4 <= nums[i] <= 10^4`.

**Follow up:** bạn có thể thiết kế thuật toán chạy trong `O(n log n)` không?

In [1]:
from typing import List
from bisect import bisect_left, bisect_right
from functools import lru_cache
import random
import sys
import time

examples = [
    ([10, 9, 2, 5, 3, 7, 101, 18], 4),
    ([0, 1, 0, 3, 2, 3], 4),
    ([7, 7, 7, 7, 7, 7, 7], 1),
]
for nums, expected in examples:
    print(f"nums={nums} -> kỳ vọng {expected}")

nums=[10, 9, 2, 5, 3, 7, 101, 18] -> kỳ vọng 4
nums=[0, 1, 0, 3, 2, 3] -> kỳ vọng 4
nums=[7, 7, 7, 7, 7, 7, 7] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Dãy con (subsequence) khác mảng con (subarray)

Với `[10, 9, 2, 5, 3]`: `[2, 5]` là **mảng con** (liền kề) cũng là dãy con; `[10, 2, 3]` là **dãy con** (bỏ `9` và `5`, giữ thứ tự) nhưng **không** là mảng con. LIS tìm **dãy con** nên có thể "nhảy cóc".
#### Code demo

In [2]:
from itertools import combinations
nums = [10, 9, 2, 5, 3]
subsequences = [list(c) for r in range(1, 4) for c in combinations(nums, r)]       # combinations giữ thứ tự gốc
print([10, 2, 3] in subsequences, [3, 2] in subsequences)                         # True False (đảo thứ tự không được)

True False


### 3.2. "Tăng ngặt" nghĩa là `<` chứ không phải `<=`

Dãy `[7, 7]` **không** tăng ngặt vì `7 < 7` sai. Ví dụ 3 `[7,7,7,...]` cho đáp án `1`. Điều này quyết định chọn `bisect_left` thay vì `bisect_right` ở cách `O(n log n)`.
#### Code demo

In [3]:
def is_strictly_increasing(seq):
    return all(seq[i] < seq[i + 1] for i in range(len(seq) - 1))


print(is_strictly_increasing([2, 3, 7, 101]), is_strictly_increasing([7, 7]), is_strictly_increasing([5]))   # True False True

True False True


### 3.3. Quy hoạch động: trạng thái "kết thúc tại `i`"

Định nghĩa `dp[i]` = **độ dài LIS kết thúc đúng tại phần tử `i`** (phần tử cuối là `nums[i]`). Để tính, nhìn mọi `j < i` có `nums[j] < nums[i]`: ta nối `nums[i]` vào LIS kết thúc tại `j`: `dp[i] = 1 + max(dp[j])` (hoặc `1` nếu không có `j`). Đáp án là `max(dp)` (LIS có thể kết thúc ở bất kỳ đâu).
#### Code demo

In [4]:
nums = [10, 9, 2, 5, 3, 7, 101, 18]
dp = [1] * len(nums)
for i in range(len(nums)):
    for j in range(i):
        if nums[j] < nums[i]:
            dp[i] = max(dp[i], dp[j] + 1)
print(dp, "-> đáp án = max(dp) =", max(dp))        # [1, 1, 1, 2, 2, 3, 4, 4] -> 4

[1, 1, 1, 2, 2, 3, 4, 4] -> đáp án = max(dp) = 4


### 3.4. Chặt nhị phân với `bisect_left` và `bisect_right`

Cho danh sách **đã sắp tăng** `a`: `bisect_left(a, x)` trả chỉ số **đầu tiên** `p` mà `a[p] >= x` (vị trí chèn `x` **trước** các phần tử bằng `x`); `bisect_right(a, x)` trả chỉ số đầu tiên mà `a[p] > x` (chèn **sau** các phần tử bằng `x`). Chạy `O(log n)`.
#### Code demo

In [5]:
a = [2, 3, 7, 18]
print(bisect_left(a, 7), bisect_right(a, 7))        # 2 3: khác nhau khi x đã có trong a
print(bisect_left(a, 5), bisect_right(a, 5))        # 2 2: giống nhau khi x chưa có
print(bisect_left(a, 1), bisect_left(a, 100))       # 0 4: đầu / cuối

2 3
2 2
0 4


### 3.5. Mảng `tails` (patience sorting)

Giữ một danh sách `tails` mà `tails[k]` là **giá trị cuối nhỏ nhất** của một dãy con tăng ngặt có độ dài `k + 1` (trong các phần tử đã xét). Nếu xử lý `x` mới: thay `tails[p]` bằng `x` với `p = bisect_left(tails, x)` (làm đuôi nhỏ hơn cho độ dài `p+1`), hoặc **thêm** `x` vào cuối nếu `x` lớn hơn mọi đuôi (LIS dài thêm `1`). Độ dài LIS là `len(tails)`.
#### Code demo

In [6]:
tails = []
for x in [10, 9, 2, 5, 3, 7, 101, 18]:
    p = bisect_left(tails, x)
    if p == len(tails):
        tails.append(x)
    else:
        tails[p] = x
    print(f"x={x:>3}: p={p} -> tails={tails}")
print("độ dài LIS =", len(tails))

x= 10: p=0 -> tails=[10]
x=  9: p=0 -> tails=[9]
x=  2: p=0 -> tails=[2]
x=  5: p=1 -> tails=[2, 5]
x=  3: p=1 -> tails=[2, 3]
x=  7: p=2 -> tails=[2, 3, 7]
x=101: p=3 -> tails=[2, 3, 7, 101]
x= 18: p=3 -> tails=[2, 3, 7, 18]
độ dài LIS = 4


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 1 `[10, 9, 2, 5, 3, 7, 101, 18]`. Dùng DP "kết thúc tại `i`":

| `i` | `nums[i]` | các `j < i` với `nums[j] < nums[i]` | `dp[i]` |
|---|---|---|---|
| 0 | 10 | — | 1 |
| 1 | 9 | — | 1 |
| 2 | 2 | — | 1 |
| 3 | 5 | `j=2` (2) | `1 + 1 = 2` |
| 4 | 3 | `j=2` (2) | `2` |
| 5 | 7 | `j=2,3,4` (2, 5, 3) | `1 + max(1,2,2) = 3` |
| 6 | 101 | mọi `j` trước | `1 + max(1,1,1,2,2,3) = 4` |
| 7 | 18 | `j=0..5` | `1 + max(...,3) = 4` |

`max(dp) = 4`. **Cách nghĩ:** để biết dãy con tăng dài nhất kết thúc tại `i`, chỉ cần biết dãy con tăng dài nhất kết thúc tại mỗi phần tử **nhỏ hơn** đứng trước nó.

## 5. Cách 1 — Brute Force / baseline: thử mọi dãy con

**Ý tưởng.** Với `n` phần tử có `2^n` dãy con (mỗi phần tử giữ hoặc bỏ). Với mỗi tập con (bằng mặt nạ bit), lấy dãy theo thứ tự gốc, kiểm tra tăng ngặt, giữ độ dài lớn nhất. Chỉ dùng được cho `n ≲ 18`.
#### Code demo

In [7]:
def lis_bruteforce(nums):
    n = len(nums)
    best = 0
    for mask in range(1 << n):
        seq = [nums[i] for i in range(n) if mask >> i & 1]
        if all(seq[k] < seq[k + 1] for k in range(len(seq) - 1)):
            best = max(best, len(seq))
    return best


for nums, expected in examples:
    print(nums, "->", lis_bruteforce(nums), "(kỳ vọng", expected, ")")

[10, 9, 2, 5, 3, 7, 101, 18] -> 4 (kỳ vọng 4 )
[0, 1, 0, 3, 2, 3] -> 4 (kỳ vọng 4 )
[7, 7, 7, 7, 7, 7, 7] -> 1 (kỳ vọng 1 )


**Độ phức tạp (Cách 1).** **Time `O(2^n · n)`**, với `n = 2500` là bất khả thi; **Space `O(n)`**.

**Nút thắt.** Rất nhiều dãy con có chung "đoạn đầu" và chỉ khác phần đuôi; ta kiểm tra lại từ đầu mỗi lần. DP chỉ tính một lần cho mỗi "kết thúc".

## 6. Ý tưởng thuật toán / bước đột phá

**DP `O(n²)`:** `dp[i] = 1 + max{dp[j] : j < i, nums[j] < nums[i]}`, đáp án `max(dp)`.

**Cải tiến `O(n log n)`:** nhận xét rằng với **mỗi độ dài `k`**, ta chỉ cần nhớ **đuôi nhỏ nhất** có thể (đuôi càng nhỏ càng dễ nối thêm). Gọi `tails[k]` là đuôi nhỏ nhất của dãy con tăng ngặt độ dài `k + 1`. `tails` **tăng ngặt** (nếu có dãy dài `k+2` kết thúc ở `v` thì cắt đuôi ta có dãy dài `k+1` với đuôi `< v`). Do đó với phần tử mới `x`, vị trí cần cập nhật tìm được bằng **chặt nhị phân**.

**Cải tiến khác:** dùng **cây Fenwick** lấy `max dp` trên đoạn giá trị `< x` trong `O(log n)`.

## 7. Các cách tiếp theo

### Cách 2 — Đệ quy có nhớ (top-down) `f(i, prev)`

**Giải thích.** `f(i, prev)` = độ dài LIS lớn nhất khi xét từ chỉ số `i` trở đi mà phần tử đã chọn cuối cùng có chỉ số `prev` (hoặc `-1`). Chọn `nums[i]` nếu `prev == -1` hoặc `nums[prev] < nums[i]`, hoặc bỏ qua.
#### Code demo

In [8]:
def lis_memo(nums):
    n = len(nums)
    sys.setrecursionlimit(max(sys.getrecursionlimit(), 10_000))

    @lru_cache(maxsize=None)
    def f(i, prev):
        if i == n:
            return 0
        best = f(i + 1, prev)                                          # bỏ nums[i]
        if prev == -1 or nums[prev] < nums[i]:
            best = max(best, 1 + f(i + 1, i))                          # chọn nums[i]
        return best

    result = f(0, -1)
    f.cache_clear()
    return result


for nums, expected in examples:
    print(nums, "->", lis_memo(nums), "(kỳ vọng", expected, ")")

[10, 9, 2, 5, 3, 7, 101, 18] -> 4 (kỳ vọng 4 )
[0, 1, 0, 3, 2, 3] -> 4 (kỳ vọng 4 )
[7, 7, 7, 7, 7, 7, 7] -> 1 (kỳ vọng 1 )


**Độ phức tạp Cách 2:** `O(n²)` trạng thái `(i, prev)`, mỗi trạng thái `O(1)`: **Time `O(n²)`**, **Space `O(n²)`** (bộ nhớ đệm) và ngăn xếp đệ quy sâu `n` (với `n = 2500` cần nâng giới hạn đệ quy).

### Cách 3 — Quy hoạch động bottom-up `dp[i]`

**Giải thích.** Như mục 3.3.
#### Code demo

In [9]:
def lis_dp(nums):
    n = len(nums)
    dp = [1] * n                                   # mỗi phần tử tự nó là một dãy độ dài 1
    for i in range(n):
        for j in range(i):
            if nums[j] < nums[i]:                  # có thể nối nums[i] vào LIS kết thúc tại j
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)


for nums, expected in examples:
    print(nums, "->", lis_dp(nums), "(kỳ vọng", expected, ")")

[10, 9, 2, 5, 3, 7, 101, 18] -> 4 (kỳ vọng 4 )
[0, 1, 0, 3, 2, 3] -> 4 (kỳ vọng 4 )
[7, 7, 7, 7, 7, 7, 7] -> 1 (kỳ vọng 1 )


**Độ phức tạp Cách 3:** **Time `O(n²)`** (`n(n-1)/2` so sánh; `n = 2500` là `~3·10^6`), **Space `O(n)`**.

### Cách 4 — `tails` + `bisect_left` (cách nộp tối ưu)

**Giải thích.** Như mục 3.5. Mỗi phần tử một lần chặt nhị phân.
#### Code demo

In [10]:
def lis_tails(nums):
    tails = []
    for x in nums:
        p = bisect_left(tails, x)                  # vị trí đầu tiên có tails[p] >= x
        if p == len(tails):
            tails.append(x)                        # x lớn hơn mọi đuôi: kéo dài LIS thêm 1
        else:
            tails[p] = x                           # thay đuôi bằng giá trị nhỏ hơn (hoặc bằng)
    return len(tails)


for nums, expected in examples:
    print(nums, "->", lis_tails(nums), "(kỳ vọng", expected, ")")

[10, 9, 2, 5, 3, 7, 101, 18] -> 4 (kỳ vọng 4 )
[0, 1, 0, 3, 2, 3] -> 4 (kỳ vọng 4 )
[7, 7, 7, 7, 7, 7, 7] -> 1 (kỳ vọng 1 )


In [11]:
# Code nộp LeetCode cho Cách 4
class Solution:
    def lengthOfLIS(self, nums: List[int]) -> int:
        d = []
        for x in nums:
            idx = bisect_left(d, x)
            if idx == len(d):
                d.append(x)
            else:
                d[idx] = x
        return len(d)


print(Solution().lengthOfLIS([10, 9, 2, 5, 3, 7, 101, 18]))   # 4

4


**Độ phức tạp Cách 4:** **Time `O(n log n)`** (`n` lần `bisect_left` `O(log n)`), **Space `O(n)`** worst case (`tails` dài tới `n`). Với `n = 2500`: cỡ `3·10^4` thao tác.

### Cách 5 — Cây Fenwick (BIT) lấy `max` trên giá trị đã nén

**Giải thích.** Nén giá trị về hạng `1..m` (sắp xếp rồi bỏ trùng). Duyệt từng `x`: `best = query(rank(x) - 1) + 1` với `query(r)` trả `max dp` của các giá trị có hạng `<= r` (tức các giá trị **nhỏ hơn `x`** đã xuất hiện trước); rồi `update(rank(x), best)`. BIT hỗ trợ `max` theo tiền tố vì ta chỉ **tăng** giá trị.
#### Code demo

In [12]:
def lis_fenwick(nums):
    ranks = {v: i + 1 for i, v in enumerate(sorted(set(nums)))}      # nén giá trị: 1..m
    m = len(ranks)
    tree = [0] * (m + 1)

    def update(i, value):                                            # tree[i] = max(tree[i], value) lan lên
        while i <= m:
            if tree[i] < value:
                tree[i] = value
            i += i & -i

    def query(i):                                                    # max trên tiền tố 1..i
        best = 0
        while i > 0:
            best = max(best, tree[i])
            i -= i & -i
        return best

    answer = 0
    for x in nums:
        r = ranks[x]
        length = query(r - 1) + 1                                    # LIS kết thúc tại x = 1 + max trên các giá trị < x
        update(r, length)
        answer = max(answer, length)
    return answer


for nums, expected in examples:
    print(nums, "->", lis_fenwick(nums), "(kỳ vọng", expected, ")")

[10, 9, 2, 5, 3, 7, 101, 18] -> 4 (kỳ vọng 4 )
[0, 1, 0, 3, 2, 3] -> 4 (kỳ vọng 4 )
[7, 7, 7, 7, 7, 7, 7] -> 1 (kỳ vọng 1 )


**Độ phức tạp Cách 5:** **Time `O(n log n)`** (sắp xếp để nén + `n` lần `query`/`update` `O(log m)`), **Space `O(n)`**. Hữu ích khi cần biến thể có **trọng số** (tổng lớn nhất dãy con tăng) hoặc đếm số LIS.

> Cách 4 là bản nộp tối ưu; Cách 3 là bản dễ nghĩ ra nhất; Cách 5 mở rộng được cho nhiều biến thể.

## 8. TRACE từng bước

Ta trace `tails` (Cách 4) trên `[10, 9, 2, 5, 3, 7, 101, 18]` và trên `[0, 1, 0, 3, 2, 3]`, ghi rõ `p` và hành động; rồi trace DP (Cách 3).
#### Code demo

In [13]:
def trace_tails(nums):
    print("nums =", nums)
    tails = []
    for x in nums:
        p = bisect_left(tails, x)
        action = "thêm vào cuối (LIS dài thêm 1)" if p == len(tails) else f"thay tails[{p}] = {tails[p]} bằng {x}"
        if p == len(tails):
            tails.append(x)
        else:
            tails[p] = x
        print(f"  x={x:>3}: bisect_left -> p={p}: {action:<36} tails={tails}")
    print("  độ dài LIS =", len(tails))
    return len(tails)


trace_tails([10, 9, 2, 5, 3, 7, 101, 18])
print()
trace_tails([0, 1, 0, 3, 2, 3])

nums = [10, 9, 2, 5, 3, 7, 101, 18]
  x= 10: bisect_left -> p=0: thêm vào cuối (LIS dài thêm 1)       tails=[10]
  x=  9: bisect_left -> p=0: thay tails[0] = 10 bằng 9            tails=[9]
  x=  2: bisect_left -> p=0: thay tails[0] = 9 bằng 2             tails=[2]
  x=  5: bisect_left -> p=1: thêm vào cuối (LIS dài thêm 1)       tails=[2, 5]
  x=  3: bisect_left -> p=1: thay tails[1] = 5 bằng 3             tails=[2, 3]
  x=  7: bisect_left -> p=2: thêm vào cuối (LIS dài thêm 1)       tails=[2, 3, 7]
  x=101: bisect_left -> p=3: thêm vào cuối (LIS dài thêm 1)       tails=[2, 3, 7, 101]
  x= 18: bisect_left -> p=3: thay tails[3] = 101 bằng 18          tails=[2, 3, 7, 18]
  độ dài LIS = 4

nums = [0, 1, 0, 3, 2, 3]
  x=  0: bisect_left -> p=0: thêm vào cuối (LIS dài thêm 1)       tails=[0]
  x=  1: bisect_left -> p=1: thêm vào cuối (LIS dài thêm 1)       tails=[0, 1]
  x=  0: bisect_left -> p=0: thay tails[0] = 0 bằng 0             tails=[0, 1]
  x=  3: bisect_left -> p=2: thêm vào cuối (

4

In [14]:
def trace_dp(nums):
    n = len(nums)
    dp = [1] * n
    print("nums =", nums)
    for i in range(n):
        picks = []
        for j in range(i):
            if nums[j] < nums[i] and dp[j] + 1 > dp[i]:
                dp[i] = dp[j] + 1
                picks.append(f"j={j}({nums[j]}) -> dp={dp[i]}")
        print(f"  i={i} (x={nums[i]}): {', '.join(picks) if picks else 'không có j nhỏ hơn'}; dp[i]={dp[i]}")
    print("  dp =", dp, "-> đáp án", max(dp))
    return max(dp)


trace_dp([10, 9, 2, 5, 3, 7, 101, 18])

nums = [10, 9, 2, 5, 3, 7, 101, 18]
  i=0 (x=10): không có j nhỏ hơn; dp[i]=1
  i=1 (x=9): không có j nhỏ hơn; dp[i]=1
  i=2 (x=2): không có j nhỏ hơn; dp[i]=1
  i=3 (x=5): j=2(2) -> dp=2; dp[i]=2
  i=4 (x=3): j=2(2) -> dp=2; dp[i]=2
  i=5 (x=7): j=2(2) -> dp=2, j=3(5) -> dp=3; dp[i]=3
  i=6 (x=101): j=0(10) -> dp=2, j=3(5) -> dp=3, j=5(7) -> dp=4; dp[i]=4
  i=7 (x=18): j=0(10) -> dp=2, j=3(5) -> dp=3, j=5(7) -> dp=4; dp[i]=4
  dp = [1, 1, 1, 2, 2, 3, 4, 4] -> đáp án 4


4

Bảng `tails` (ví dụ 1):

| `x` | `p` | hành động | `tails` |
|---|---|---|---|
| 10 | 0 | thêm | `[10]` |
| 9 | 0 | thay `10` bằng `9` | `[9]` |
| 2 | 0 | thay `9` bằng `2` | `[2]` |
| 5 | 1 | thêm | `[2, 5]` |
| 3 | 1 | thay `5` bằng `3` | `[2, 3]` |
| 7 | 2 | thêm | `[2, 3, 7]` |
| 101 | 3 | thêm | `[2, 3, 7, 101]` |
| 18 | 3 | thay `101` bằng `18` | `[2, 3, 7, 18]` |

Độ dài `tails = 4`. Ở ví dụ này `tails` cuối `[2, 3, 7, 18]` tình cờ cũng là một dãy con tăng hợp lệ, nhưng nói chung `tails` **không** phải một dãy con của `nums` (xem Lỗi 5), chỉ có độ dài `len(tails)` là đáp án.

## 9. Code nộp LeetCode

Bản nộp dùng `tails` + `bisect_left` (Cách 4). Đúng `class Solution`, đúng `lengthOfLIS(self, nums: List[int]) -> int`.

In [15]:
class Solution:
    def lengthOfLIS(self, nums: List[int]) -> int:
        tails = []                                   # tails[k] = đuôi nhỏ nhất của một dãy con tăng ngặt độ dài k + 1
        for x in nums:
            p = bisect_left(tails, x)                # bisect_left vì cần tăng NGẶT (x bằng một đuôi thì thay chỗ đó)
            if p == len(tails):
                tails.append(x)
            else:
                tails[p] = x
        return len(tails)


sol = Solution()
for nums, expected in examples:
    assert sol.lengthOfLIS(nums) == expected
print("OK: 3 ví dụ của đề")

OK: 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **DP `O(n²)`:** mọi dãy con tăng kết thúc tại `i` có độ dài `>= 2` đều có phần tử áp chót `nums[j]` với `j < i`, `nums[j] < nums[i]`, và phần đứng trước là dãy con tăng kết thúc tại `j`; để dài nhất ta chọn dãy dài nhất ⇒ `dp[i] = 1 + max dp[j]`. Đáp án = `max dp` vì LIS kết thúc ở một chỉ số nào đó.
- **Invariant của `tails`:** sau khi xử lý các phần tử `x_0..x_t`, với mọi `k`, `tails[k]` bằng đuôi nhỏ nhất trong mọi dãy con tăng ngặt độ dài `k + 1` của `x_0..x_t` (và `len(tails)` = độ dài LIS). Chứng minh quy nạp: thêm `x`: (a) `tails` tăng ngặt nên `p = bisect_left(tails, x)` là vị trí nhỏ nhất `p` mà `tails[p] >= x`; (b) với mọi `k < p` có `tails[k] < x`, nên `x` nối được vào dãy độ dài `k+1` ⇒ tồn tại dãy độ dài `p + 1` kết thúc ở `x` (nối vào dãy độ dài `p` có đuôi `tails[p-1] < x`); (c) vì `tails[p] >= x`, `x` là đuôi **nhỏ hơn hoặc bằng** đuôi cũ của độ dài `p+1` nên ta cập nhật `tails[p] = x` (nếu bằng thì không đổi); (d) không dãy độ dài `p + 2` nào có đuôi cập nhật được vì cần đuôi cũ độ dài `p+1` `< x`, mà `tails[p] >= x`.
- **`tails` tăng ngặt:** nếu tồn tại dãy tăng ngặt độ dài `k+2` đuôi `v`, bỏ phần tử cuối ta được dãy độ dài `k+1` có đuôi `< v` ⇒ `tails[k] < tails[k+1]`.
- **`bisect_left` ứng với tăng ngặt:** nếu `x == tails[p]` ta **thay** (không thêm), vì `x` không thể nối sau chính nó. Dùng `bisect_right` sẽ đặt `x` sau các đuôi bằng nó ⇒ cho phép dãy không giảm (sai).
- **Cách 5:** `query(rank(x)-1)` cho max `dp` của các giá trị **nhỏ hơn** `x` đã gặp trước (đúng điều kiện nối); `update` ghi nhận `dp` cho giá trị `x`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)` (tối đa `2500`).

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn mọi dãy con | `O(2^n·n)` | `O(n)` |
| 2. Đệ quy có nhớ | `O(n²)` | `O(n²)` |
| 3. DP `dp[i]` | `O(n²)` | `O(n)` |
| 4. `tails` + `bisect_left` | `O(n log n)` | `O(n)` |
| 5. Fenwick | `O(n log n)` | `O(n)` |

Với `n = 2500`: Cách 3 cỡ `3·10^6`, Cách 4 cỡ `3·10^4`. Với `n = 10^5` (không thuộc đề) chỉ Cách 4/5 còn dùng được. Không có khác biệt average/worst về bậc (dữ liệu chỉ ảnh hưởng hằng số: `tails` ngắn khi LIS ngắn).

## 12. Bộ test

`oracle` là brute force độc lập: **thử mọi tập con** bằng mặt nạ bit (như Cách 1 nhưng kiểm tra tăng ngặt bằng đối chiếu với `sorted(set(seq))`), dùng cho `n <= 14`. Với `n` lớn, đối chiếu các cách `O(n²)` và `O(n log n)` với nhau.
#### Code demo

In [16]:
def oracle(nums):
    n = len(nums)
    best = 0
    for mask in range(1 << n):
        seq = [nums[i] for i in range(n) if mask >> i & 1]
        if seq == sorted(set(seq)):                       # tăng ngặt ⟺ bằng chính bản sắp xếp bỏ trùng
            best = max(best, len(seq))
    return best


impls = {
    "bruteforce": lis_bruteforce,
    "memo": lis_memo,
    "dp": lis_dp,
    "tails": lis_tails,
    "fenwick": lis_fenwick,
    "submit": sol.lengthOfLIS,
}

for nums, expected in examples:
    assert oracle(nums) == expected
    for key, fn in impls.items():
        assert fn(list(nums)) == expected, (key, nums, fn(list(nums)), expected)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên
#### Code demo

In [17]:
edge_cases = [
    ([5], 1, "một phần tử"),
    ([1, 2], 2, "tăng"),
    ([2, 1], 1, "giảm"),
    ([1, 1], 1, "hai phần tử bằng nhau: không tăng ngặt"),
    ([1, 2, 3, 4, 5], 5, "tăng dần toàn bộ"),
    ([5, 4, 3, 2, 1], 1, "giảm dần"),
    ([7, 7, 7, 7], 1, "toàn bằng nhau"),
    ([1, 3, 2, 4, 3, 5], 4, "zigzag: 1,3,4,5 hoặc 1,2,3,5"),
    ([-10000, 10000], 2, "giá trị biên của đề"),
    ([3, 10, 2, 1, 20], 3, "LIS 3,10,20"),
    ([0, 1, 0, 3, 2, 3], 4, "ví dụ 2"),
    ([2, 2, 3, 3, 4, 4], 3, "nhiều phần tử trùng: chỉ tính một bản mỗi giá trị"),
]
for nums, expected, label in edge_cases:
    assert oracle(nums) == expected, (label, oracle(nums), expected)
    for key, fn in impls.items():
        assert fn(list(nums)) == expected, (key, label, fn(list(nums)), expected)
    print("PASS", label)

PASS một phần tử
PASS tăng
PASS giảm
PASS hai phần tử bằng nhau: không tăng ngặt
PASS tăng dần toàn bộ
PASS giảm dần
PASS toàn bằng nhau
PASS zigzag: 1,3,4,5 hoặc 1,2,3,5
PASS giá trị biên của đề
PASS LIS 3,10,20
PASS ví dụ 2
PASS nhiều phần tử trùng: chỉ tính một bản mỗi giá trị


### 12.2. Tính chất và phản ví dụ

- `1 <= đáp án <= n`; mảng tăng ngặt đáp án `n`; mảng không tăng (giảm hoặc bằng nhau) đáp án `1`.
- **Đảo ngược mảng và đổi dấu** (`nums[::-1]` với `x → -x`) không đổi LIS (đối xứng).
- Thêm một số **lớn hơn mọi phần tử** vào cuối tăng LIS đúng `1`.
- **Phản ví dụ** cho `bisect_right`: `[7,7,7,7]` cho `4` (sai), đúng `1`.
- **Phản ví dụ** cho "mảng con": `[1,3,2,4]` có LIS (dãy con) `3` nhưng mảng con tăng dài nhất chỉ `2`.
#### Code demo

In [18]:
rng = random.Random(300)
for _ in range(400):
    n = rng.randint(1, 40)
    nums = [rng.randint(-20, 20) for _ in range(n)]
    ans = sol.lengthOfLIS(nums)
    assert 1 <= ans <= n
    assert sol.lengthOfLIS([-x for x in nums[::-1]]) == ans                  # đảo + đổi dấu
    assert sol.lengthOfLIS(nums + [max(nums) + 1]) == ans + 1
print("PASS tính chất (cận, đối xứng đảo+đổi dấu, thêm số lớn nhất) trên 400 mảng ngẫu nhiên")
assert sol.lengthOfLIS(list(range(30))) == 30 and sol.lengthOfLIS(list(range(30, 0, -1))) == 1


def sai_bisect_right(nums):
    tails = []
    for x in nums:
        p = bisect_right(tails, x)                                            # SAI: cho phép bằng nhau
        if p == len(tails):
            tails.append(x)
        else:
            tails[p] = x
    return len(tails)


print("phản ví dụ [7,7,7,7]: đúng", sol.lengthOfLIS([7, 7, 7, 7]), "| bisect_right:", sai_bisect_right([7, 7, 7, 7]))
assert sai_bisect_right([7, 7, 7, 7]) == 4


def longest_increasing_subarray(nums):
    best = cur = 1
    for i in range(1, len(nums)):
        cur = cur + 1 if nums[i] > nums[i - 1] else 1
        best = max(best, cur)
    return best


print("[1,3,2,4]: LIS (dãy con) =", sol.lengthOfLIS([1, 3, 2, 4]), "| mảng con tăng dài nhất =", longest_increasing_subarray([1, 3, 2, 4]))

PASS tính chất (cận, đối xứng đảo+đổi dấu, thêm số lớn nhất) trên 400 mảng ngẫu nhiên
phản ví dụ [7,7,7,7]: đúng 1 | bisect_right: 4
[1,3,2,4]: LIS (dãy con) = 3 | mảng con tăng dài nhất = 2


### 12.3. Đối chiếu ngẫu nhiên (có seed) và `n` lớn
#### Code demo

In [19]:
rng = random.Random(4300)
for trial in range(2500):
    n = rng.randint(1, 12)
    span = rng.choice([2, 5, 20, 10_000])
    nums = [rng.randint(-span, span) for _ in range(n)]
    truth = oracle(nums)
    for key, fn in impls.items():
        assert fn(list(nums)) == truth, (key, nums, fn(list(nums)), truth)
print("PASS 2500 mảng ngẫu nhiên nhỏ (nhiều phần tử trùng): mọi cách khớp oracle")

# n = 2500 (cỡ tối đa của đề): bỏ vét cạn và oracle; so các cách còn lại
for name, nums in [("ngẫu nhiên", [rng.randint(-10000, 10000) for _ in range(2500)]),
                   ("tăng dần", list(range(2500))),
                   ("giảm dần", list(range(2500, 0, -1))),
                   ("toàn bằng nhau", [5] * 2500),
                   ("zigzag", [(-1) ** i * i for i in range(2500)]),
                   ("ít giá trị khác nhau", [rng.randint(0, 5) for _ in range(2500)])]:
    row = []
    ref = None
    for key in ("dp", "tails", "fenwick", "submit"):
        t0 = time.perf_counter()
        value = impls[key](list(nums))
        dt = (time.perf_counter() - t0) * 1000
        ref = value if ref is None else ref
        assert value == ref, (name, key)
        row.append(f"{key}={dt:7.1f}ms")
    print(f"{name:<22} LIS={ref:>5} | " + " ".join(row))
small = [rng.randint(-10000, 10000) for _ in range(300)]
assert lis_memo(small) == lis_tails(small)
print("PASS n = 2500 (Cách 2 đệ quy có nhớ kiểm tra riêng ở n = 300 vì tốn bộ nhớ O(n^2))")

# n = 10^5 (ngoài đề): chỉ O(n log n)
big = [rng.randint(-10 ** 9, 10 ** 9) for _ in range(100_000)]
t0 = time.perf_counter()
a, b = lis_tails(big), lis_fenwick(big)
assert a == b
print(f"n = 10^5: LIS = {a} (tails + Fenwick khớp nhau), {(time.perf_counter() - t0) * 1000:.0f} ms")

PASS 2500 mảng ngẫu nhiên nhỏ (nhiều phần tử trùng): mọi cách khớp oracle
ngẫu nhiên             LIS=   91 | dp=  147.0ms tails=    0.3ms fenwick=    3.2ms submit=    0.3ms


tăng dần               LIS= 2500 | dp=  206.7ms tails=    0.2ms fenwick=    2.7ms submit=    0.2ms
giảm dần               LIS=    1 | dp=   77.1ms tails=    0.1ms fenwick=    2.4ms submit=    0.1ms
toàn bằng nhau         LIS=    1 | dp=   72.1ms tails=    0.1ms fenwick=    0.4ms submit=    0.1ms


zigzag                 LIS= 1250 | dp=  135.9ms tails=    0.2ms fenwick=    2.7ms submit=    0.2ms
ít giá trị khác nhau   LIS=    6 | dp=  134.0ms tails=    0.2ms fenwick=    0.7ms submit=    0.2ms
PASS n = 2500 (Cách 2 đệ quy có nhớ kiểm tra riêng ở n = 300 vì tốn bộ nhớ O(n^2))


n = 10^5: LIS = 617 (tails + Fenwick khớp nhau), 247 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `bisect_right` (cho phép phần tử bằng nhau)
Xem `sai_bisect_right` ở phần test: `[7,7,7,7]` cho `4`. **Vì sao sai:** đề yêu cầu **tăng ngặt**; `x` bằng một đuôi phải **thay** đuôi đó, không được nối sau. **Code đúng:** `bisect_left`.

### Lỗi 2: Nhầm dãy con với mảng con
Xem `longest_increasing_subarray` ở phần test: `[1,3,2,4]` có mảng con tăng dài nhất `2` nhưng LIS dãy con là `3`. **Code đúng:** `dp[i]` xét **mọi** `j < i`, không chỉ `j = i - 1`.

### Lỗi 3: Khởi tạo `dp` bằng `0` (hoặc quên `+1`)
#### Code sai

In [20]:
def sai_dp_khoi_tao_0(nums):
    dp = [0] * len(nums)                                   # SAI: mỗi phần tử tự nó đã là dãy độ dài 1
    for i in range(len(nums)):
        for j in range(i):
            if nums[j] < nums[i]:
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)


for nums in ([10, 9, 2, 5, 3, 7, 101, 18], [5]):
    print(nums, "đúng:", sol.lengthOfLIS(nums), "| sai:", sai_dp_khoi_tao_0(nums))

[10, 9, 2, 5, 3, 7, 101, 18] đúng: 4 | sai: 3
[5] đúng: 1 | sai: 0


**Vì sao sai:** nếu không có `j` nào thì `dp[i]` phải là `1` (chính `nums[i]`), không phải `0`. Mọi giá trị bị thiếu `1`.

### Lỗi 4: Trả `dp[-1]` thay vì `max(dp)`
#### Code sai

In [21]:
def sai_tra_dp_cuoi(nums):
    dp = [1] * len(nums)
    for i in range(len(nums)):
        for j in range(i):
            if nums[j] < nums[i]:
                dp[i] = max(dp[i], dp[j] + 1)
    return dp[-1]                                          # SAI: LIS có thể kết thúc ở bất kỳ vị trí nào


for nums in ([10, 9, 2, 5, 3, 7, 101, 18], [1, 2, 3, 0]):
    print(nums, "đúng:", sol.lengthOfLIS(nums), "| sai:", sai_tra_dp_cuoi(nums))

[10, 9, 2, 5, 3, 7, 101, 18] đúng: 4 | sai: 4
[1, 2, 3, 0] đúng: 3 | sai: 1


**Vì sao sai:** `[1,2,3,0]` có `dp = [1,2,3,1]`, `dp[-1] = 1` nhưng LIS là `3` (kết thúc ở `3`). Phải lấy `max(dp)`.

### Lỗi 5: Coi `tails` là một LIS thật
#### Code sai

In [22]:
def lay_tails(nums):
    tails = []
    for x in nums:
        p = bisect_left(tails, x)
        if p == len(tails):
            tails.append(x)
        else:
            tails[p] = x
    return tails


def is_subsequence(sub, seq):
    it = iter(seq)
    return all(x in it for x in sub)                     # tiêu thụ iterator nên giữ đúng thứ tự


nums = [4, 5, 6, 1, 2]
t = lay_tails(nums)
print("tails =", t, "-> độ dài", len(t), "(đúng là 3)")
print("tails có phải dãy con của nums (theo đúng thứ tự)?", is_subsequence(t, nums))

tails = [1, 2, 6] -> độ dài 3 (đúng là 3)
tails có phải dãy con của nums (theo đúng thứ tự)? False


**Vì sao sai:** `tails = [1, 2, 6]` nhưng `1, 2` đứng **sau** `6` trong `nums`, nên `[1, 2, 6]` **không** là dãy con theo thứ tự gốc. `tails` chỉ cho **độ dài** đúng; muốn dựng chính dãy phải lưu thêm con trỏ cha (xem phần biến thể).

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**dãy con tăng / giảm dài nhất**" → `dp[i]` kết thúc tại `i` (`O(n²)`) hoặc **`tails` + `bisect`** (`O(n log n)`).
- **Tăng ngặt ⇒ `bisect_left`**; **không giảm ⇒ `bisect_right`**.
- Muốn **dựng lại** dãy: lưu `parent[i]` (DP) hoặc lưu chỉ số vào `tails` kèm cha.
- Muốn **đếm** số LIS: DP thêm `count[i]` (LeetCode 673) hoặc Fenwick đếm.
- Biến bài toán về LIS: **Russian Doll Envelopes** (354, sắp theo `w` tăng và `h` giảm), **Maximum Height by Stacking Cuboids**, **Longest Chain**, **Minimum Deletions to Make Array Sorted** (`n - LIS`).
- Họ hàng: **Longest Common Subsequence** (1143), **Number of LIS** (673), **Increasing Triplet Subsequence** (334).
- Luôn thử: mảng tăng, giảm, toàn bằng nhau, zigzag.

## 15. Biến thể của bài

1. **Dựng lại LIS** (không chỉ độ dài).
2. **Number of Longest Increasing Subsequence** (LeetCode 673): đếm số LIS.
3. **Increasing Triplet Subsequence** (334): có dãy con tăng độ dài `3` không (`O(n)`, `O(1)`).
4. **Russian Doll Envelopes** (354): LIS hai chiều.
5. **Longest Non-decreasing Subsequence**: không giảm ⇒ `bisect_right`.
6. **Số phần tử tối thiểu cần xóa để mảng tăng ngặt**: `n - LIS`.
#### Code demo

In [23]:
def lis_reconstruct(nums):
    """Dựng một LIS cụ thể bằng DP + con trỏ cha."""
    n = len(nums)
    dp = [1] * n
    parent = [-1] * n
    for i in range(n):
        for j in range(i):
            if nums[j] < nums[i] and dp[j] + 1 > dp[i]:
                dp[i] = dp[j] + 1
                parent[i] = j
    end = max(range(n), key=lambda i: dp[i])
    seq = []
    while end != -1:
        seq.append(nums[end])
        end = parent[end]
    return seq[::-1]


def count_lis(nums):
    n = len(nums)
    length, count = [1] * n, [1] * n
    for i in range(n):
        for j in range(i):
            if nums[j] < nums[i]:
                if length[j] + 1 > length[i]:
                    length[i], count[i] = length[j] + 1, count[j]
                elif length[j] + 1 == length[i]:
                    count[i] += count[j]
    best = max(length)
    return sum(c for l, c in zip(length, count) if l == best)


def increasing_triplet(nums):
    first = second = float("inf")
    for x in nums:
        if x <= first:
            first = x
        elif x <= second:
            second = x
        else:
            return True
    return False


def lnds(nums):                                           # dãy con KHÔNG GIẢM dài nhất: bisect_right
    tails = []
    for x in nums:
        p = bisect_right(tails, x)
        if p == len(tails):
            tails.append(x)
        else:
            tails[p] = x
    return len(tails)


print(lis_reconstruct([10, 9, 2, 5, 3, 7, 101, 18]), count_lis([1, 3, 5, 4, 7]), increasing_triplet([2, 1, 5, 0, 4, 6]), lnds([7, 7, 7, 7]))
# [2, 5, 7, 101] hoặc tương tự; 2; True; 4

[2, 5, 7, 101] 2 True 4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi dãy con | `O(2^n·n)` | `O(n)` | oracle, `n <= 16` |
| 2. Đệ quy có nhớ | `f(i, prev)` chọn/bỏ | `O(n²)` | `O(n²)` | viết nhanh từ định nghĩa |
| 3. DP `dp[i]` | LIS kết thúc tại `i` | `O(n²)` | `O(n)` | dễ dựng lại dãy, dễ mở rộng |
| 4. `tails` + `bisect_left` | đuôi nhỏ nhất theo độ dài | `O(n log n)` | `O(n)` | **bản nộp tối ưu** |
| 5. Fenwick trên giá trị nén | max `dp` các giá trị nhỏ hơn | `O(n log n)` | `O(n)` | biến thể có trọng số/đếm |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `bisect_left([1, 3, 5], 3)` và `bisect_right([1, 3, 5], 3)`; (b) `Solution().lengthOfLIS([1, 3, 2, 4])`; (c) `Solution().lengthOfLIS([3, 2, 1])`; (d) `Solution().lengthOfLIS([1, 1, 2, 2, 3])`.

**Bài 2 (tính bằng tay).** Lập bảng `dp` cho `[3, 1, 4, 1, 5, 9, 2, 6]` và bảng `tails` tương ứng.

**Bài 3 (điền TODO).** Hoàn thiện `lis_todo(nums)` theo Cách 4.
#### Code demo

In [24]:
def lis_todo(nums):
    tails = []
    # TODO 1: duyệt x trong nums
    # TODO 2: p = bisect_left(tails, x)
    # TODO 3: nếu p == len(tails): tails.append(x)
    # TODO 4: ngược lại tails[p] = x
    # TODO 5: trả len(tails)
    pass


# Tự làm xong rồi mới mở test:
# for nums, expected in examples:
#     assert lis_todo(nums) == expected

**Bài 4 (tự viết lại).** Viết `min_deletions_to_sort(nums)` trả về số phần tử tối thiểu cần xóa để mảng còn lại **tăng ngặt**, và `longest_decreasing(nums)` trả LDS (dãy con giảm ngặt dài nhất) bằng cách đảo dấu.

**Bài 5 (biến thể / mini challenge).** Viết `russian_doll(envelopes)` (LeetCode 354): đếm số phong bì lồng nhau tối đa, mỗi phong bì `(w, h)` lồng được vào cái khác nếu **cả `w` và `h` đều nhỏ hơn ngặt**. Gợi ý: sắp theo `w` tăng, `h` **giảm** khi `w` bằng nhau, rồi LIS trên `h`. Thử `[(5,4),(6,4),(6,7),(2,3)]` → `3`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `bisect_left` đứng **trước** các số bằng; tăng **ngặt**. Bài 2: dùng bảng 3.3. Bài 5: sắp theo `w` rồi quy về LIS.
- **Mức 2 (vừa):** Bài 1: `(1, 2)`, `3`, `1`, `3`. Bài 2: `dp = [1, 1, 2, 1, 3, 4, 2, 4]`, LIS `4` (ví dụ `1,4,5,9`); `tails`: `[3]`, `[1]`, `[1,4]`, `[1,4]` (`bisect_left([1,4],1) = 0`: thay `1` bằng `1`), `[1,4,5]`, `[1,4,5,9]`, `[1,2,5,9]`, `[1,2,5,6]`. Bài 3: khung Cách 4.
- **Mức 3 (sâu):** Bài 4: `n - LIS`; LDS của `a` là LIS của `[-x for x in a]`. Bài 5: sắp `h` **giảm** với cùng `w` để hai phong bì cùng `w` không bị LIS nối với nhau; đáp án là LIS trên danh sách `h` đã sắp.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [25]:
# Bài 1
assert (bisect_left([1, 3, 5], 3), bisect_right([1, 3, 5], 3)) == (1, 2)
assert Solution().lengthOfLIS([1, 3, 2, 4]) == 3 and Solution().lengthOfLIS([3, 2, 1]) == 1
assert Solution().lengthOfLIS([1, 1, 2, 2, 3]) == 3

# Bài 2
b2 = [3, 1, 4, 1, 5, 9, 2, 6]
dp2 = [1] * len(b2)
for i in range(len(b2)):
    for j in range(i):
        if b2[j] < b2[i]:
            dp2[i] = max(dp2[i], dp2[j] + 1)
assert dp2 == [1, 1, 2, 1, 3, 4, 2, 4] and max(dp2) == 4 == oracle(b2)


# Bài 3
def lis_todo_ans(nums):
    tails = []
    for x in nums:
        p = bisect_left(tails, x)
        if p == len(tails):
            tails.append(x)
        else:
            tails[p] = x
    return len(tails)


for nums, expected in examples:
    assert lis_todo_ans(nums) == expected


# Bài 4
def min_deletions_to_sort(nums):
    return len(nums) - Solution().lengthOfLIS(nums)


def longest_decreasing(nums):
    return Solution().lengthOfLIS([-x for x in nums])


assert min_deletions_to_sort([10, 9, 2, 5, 3, 7, 101, 18]) == 4
assert longest_decreasing([5, 4, 3, 2, 1]) == 5 and longest_decreasing([1, 2, 3]) == 1


# Bài 5
def russian_doll(envelopes):
    ordered = sorted(envelopes, key=lambda e: (e[0], -e[1]))
    return Solution().lengthOfLIS([h for _, h in ordered])


assert russian_doll([(5, 4), (6, 4), (6, 7), (2, 3)]) == 3
assert russian_doll([(1, 1), (1, 1), (1, 1)]) == 1
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (`tails` + `bisect_left`)

```python
from bisect import bisect_left

class Solution:
    def lengthOfLIS(self, nums: List[int]) -> int:
        tails = []
        for x in nums:
            p = bisect_left(tails, x)
            if p == len(tails):
                tails.append(x)
            else:
                tails[p] = x
        return len(tails)
```

**Checklist**

- [ ] Hiểu **dãy con** (không liền kề) và **tăng ngặt** (`<`)?
- [ ] `tails[k]` = đuôi nhỏ nhất của dãy con tăng độ dài `k+1`; dùng **`bisect_left`**?
- [ ] Biết `tails` chỉ cho **độ dài**, không phải dãy thật?
- [ ] DP: `dp[i]` khởi tạo `1`, đáp án `max(dp)` (không phải `dp[-1]`)?
- [ ] Đã thử toàn bằng nhau (`1`), tăng dần (`n`), giảm dần (`1`), zigzag, phần tử trùng?
- [ ] Đã đối chiếu với brute force mặt nạ bit trên mảng ngẫu nhiên nhỏ?
- [ ] Skeleton đúng `class Solution` / `def lengthOfLIS(self, nums: List[int]) -> int`?